# Preprocessing sanity-check notebook

This notebook is used to validate the preprocessing logic before launching the XLM-R runs.

It checks:
- title/review splitting
- review_only vs title_plus_review
- light / medium / heavy preprocessing
- URL token preservation
- tokenization behavior

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

In [2]:
ROOT = Path.cwd()

# Climb up until we find a folder that looks like the repo root
while ROOT != ROOT.parent and not ((ROOT / 'data').exists() and (ROOT / 'deep_learning').exists()):
    ROOT = ROOT.parent

os.chdir(ROOT)
print('Working directory:', Path.cwd())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

Working directory: /Users/yousseftorgeman/Desktop/text_classifier


In [3]:

from transformers import AutoTokenizer
from deep_learning.preprocessing.preprocessor import Preprocessor
from deep_learning.code_configs import Preprocessor_Configs

/Users/yousseftorgeman/Desktop/text_classifier/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
DATA_PATH = Path('data/train.csv')
SAMPLE_N = 30

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Could not find {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
df_sample = df.sample(n=min(SAMPLE_N, len(df)), random_state=42).reset_index(drop=True)

print('Loaded sample shape:', df_sample.shape)
df_sample.head()

Loaded sample shape: (30, 3)


,id,sentence,label
0,75255,Toilettenbürste\n\nKeine schöne Verarbeitung. ...,1
1,192435,Leider ein Fehlkauf 😢\n\nHabe die Sterne vor 2...,0
2,154839,"NOT LED!\n\nUnfortunately, not led. Other than...",2
3,59774,Tun was sie sollen\n\nDie Adapter sind preiswe...,4
4,63788,Alles super\n\nProdukt ist wie beschrieben. Gu...,4


In [5]:
print(df.columns.tolist())

['id', 'sentence', 'label']


In [6]:
def display_example(title, text):
    print('=' * 100)
    print(title)
    print('=' * 100)
    print(text)
    print()

## Manual examples

In [7]:
manual_examples = [
    'Great product\n\nI absolutely loved it!!!!!!! 😊 Check https://example.com for details.',
    "Not bad at all\n\nI didn’t expect much, but it works surprisingly well!!!",
    'Terrible quality\n\nDo not buy this product!!!!! It broke after 2 days... www.badshop.com',
    'Simple title only without body',
    'Amazing deal\n\nSO GOOD!!! REALLY REALLY GOOD!!! Visit http://shop.com now!!! 😍'
]

tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

for input_format in ['review_only', 'title_plus_review']:
    for level in ['light', 'medium', 'heavy']:
        cfg = Preprocessor_Configs(
            preprocessing_level=level,
            input_format=input_format,
            truncation=True,
            max_length=128,
            padding=False,
        )
        preproc = Preprocessor(cfg, tokenizer, return_special_tokens_mask=False)

        print('#' * 120)
        print(f'INPUT FORMAT = {input_format} | PREPROCESSING LEVEL = {level}')
        print('#' * 120)

        for i, example in enumerate(manual_examples, start=1):
            processed = preproc.preprocess_one_text(example)
            display_example(f'Example {i}', processed)

########################################################################################################################
INPUT FORMAT = review_only | PREPROCESSING LEVEL = light
########################################################################################################################
Example 1
I absolutely loved it!!!!!!! 😊 Check URLTOKEN for details.

Example 2
I didn’t expect much, but it works surprisingly well!!!

Example 3
Do not buy this product!!!!! It broke after 2 days... URLTOKEN

Example 4
Simple title only without body

Example 5
SO GOOD!!! REALLY REALLY GOOD!!! Visit URLTOKEN now!!! 😍

########################################################################################################################
INPUT FORMAT = review_only | PREPROCESSING LEVEL = medium
########################################################################################################################
Example 1
I absolutely loved it!!! 😊 Check URLTOKEN for details.

Example 2
I di

## Real dataset examples

In [8]:
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

for idx in range(5):
    raw_text = df_sample.loc[idx, 'sentence']

    print('#' * 120)
    print(f'RAW EXAMPLE {idx}')
    print('#' * 120)
    print(raw_text)
    print()

    for input_format in ['review_only', 'title_plus_review']:
        for level in ['light', 'medium', 'heavy']:
            cfg = Preprocessor_Configs(
                preprocessing_level=level,
                input_format=input_format,
                truncation=True,
                max_length=128,
                padding=False,
            )
            preproc = Preprocessor(cfg, tokenizer, return_special_tokens_mask=False)
            processed = preproc.preprocess_one_text(raw_text)

            display_example(f'{input_format} | {level}', processed)

########################################################################################################################
RAW EXAMPLE 0
########################################################################################################################
Toilettenbürste

Keine schöne Verarbeitung. Etwas kitschig.

review_only | light
Keine schöne Verarbeitung. Etwas kitschig.

review_only | medium
Keine schöne Verarbeitung. Etwas kitschig.

review_only | heavy
keine schöne verarbeitung. etwas kitschig.

title_plus_review | light
Title: Toilettenbürste Review: Keine schöne Verarbeitung. Etwas kitschig.

title_plus_review | medium
Title: Toilettenbürste Review: Keine schöne Verarbeitung. Etwas kitschig.

title_plus_review | heavy
title: toilettenbürste review: keine schöne verarbeitung. etwas kitschig.

########################################################################################################################
RAW EXAMPLE 1
###################################################

## Tabular preview

In [9]:
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

rows = []

for _, row in df_sample.head(10).iterrows():
    raw_text = row['sentence']
    entry = {'raw_text': raw_text}

    for input_format in ['review_only', 'title_plus_review']:
        for level in ['light', 'medium', 'heavy']:
            cfg = Preprocessor_Configs(
                preprocessing_level=level,
                input_format=input_format,
                truncation=True,
                max_length=128,
                padding=False,
            )
            preproc = Preprocessor(cfg, tokenizer, return_special_tokens_mask=False)
            entry[f'{level}_{input_format}'] = preproc.preprocess_one_text(raw_text)

    rows.append(entry)

preview_df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 300)
preview_df

,raw_text,light_review_only,medium_review_only,heavy_review_only,light_title_plus_review,medium_title_plus_review,heavy_title_plus_review
0,Toilettenbürste\n\nKeine schöne Verarbeitung. Etwas kitschig.,Keine schöne Verarbeitung. Etwas kitschig.,Keine schöne Verarbeitung. Etwas kitschig.,keine schöne verarbeitung. etwas kitschig.,Title: Toilettenbürste Review: Keine schöne Verarbeitung. Etwas kitschig.,Title: Toilettenbürste Review: Keine schöne Verarbeitung. Etwas kitschig.,title: toilettenbürste review: keine schöne verarbeitung. etwas kitschig.
1,Leider ein Fehlkauf 😢\n\nHabe die Sterne vor 2 Tagen angebracht aber muss leider sagen da leuchtet gar nichts schade leider rausgeschmissen Geld. Nicht zu empfehlen.,Habe die Sterne vor 2 Tagen angebracht aber muss leider sagen da leuchtet gar nichts schade leider rausgeschmissen Geld. Nicht zu empfehlen.,Habe die Sterne vor 2 Tagen angebracht aber muss leider sagen da leuchtet gar nichts schade leider rausgeschmissen Geld. Nicht zu empfehlen.,habe die sterne vor 2 tagen angebracht aber muss leider sagen da leuchtet gar nichts schade leider rausgeschmissen geld. nicht zu empfehlen.,Title: Leider ein Fehlkauf 😢 Review: Habe die Sterne vor 2 Tagen angebracht aber muss leider sagen da leuchtet gar nichts schade leider rausgeschmissen Geld. Nicht zu empfehlen.,Title: Leider ein Fehlkauf 😢 Review: Habe die Sterne vor 2 Tagen angebracht aber muss leider sagen da leuchtet gar nichts schade leider rausgeschmissen Geld. Nicht zu empfehlen.,title: leider ein fehlkauf review: habe die sterne vor 2 tagen angebracht aber muss leider sagen da leuchtet gar nichts schade leider rausgeschmissen geld. nicht zu empfehlen.
2,"NOT LED!\n\nUnfortunately, not led. Other than that.. OK!","Unfortunately, not led. Other than that.. OK!","Unfortunately, not led. Other than that.. OK!","unfortunately, not led. other than that.. ok!","Title: NOT LED! Review: Unfortunately, not led. Other than that.. OK!","Title: NOT LED! Review: Unfortunately, not led. Other than that.. OK!","title: not led! review: unfortunately, not led. other than that.. ok!"
3,"Tun was sie sollen\n\nDie Adapter sind preiswert und funktionieren gut, vor allem sitzen sie auch dicht auf dem Ventil. Würde ich immer wieder kaufen.","Die Adapter sind preiswert und funktionieren gut, vor allem sitzen sie auch dicht auf dem Ventil. Würde ich immer wieder kaufen.","Die Adapter sind preiswert und funktionieren gut, vor allem sitzen sie auch dicht auf dem Ventil. Würde ich immer wieder kaufen.","die adapter sind preiswert und funktionieren gut, vor allem sitzen sie auch dicht auf dem ventil. würde ich immer wieder kaufen.","Title: Tun was sie sollen Review: Die Adapter sind preiswert und funktionieren gut, vor allem sitzen sie auch dicht auf dem Ventil. Würde ich immer wieder kaufen.","Title: Tun was sie sollen Review: Die Adapter sind preiswert und funktionieren gut, vor allem sitzen sie auch dicht auf dem Ventil. Würde ich immer wieder kaufen.","title: tun was sie sollen review: die adapter sind preiswert und funktionieren gut, vor allem sitzen sie auch dicht auf dem ventil. würde ich immer wieder kaufen."
4,"Alles super\n\nProdukt ist wie beschrieben. Gute Qualität, gute Optik. Als Zwischenlage für eine Mehrstöckige Torte würde ich dies jedoch nicht verwenden.","Produkt ist wie beschrieben. Gute Qualität, gute Optik. Als Zwischenlage für eine Mehrstöckige Torte würde ich dies jedoch nicht verwenden.","Produkt ist wie beschrieben. Gute Qualität, gute Optik. Als Zwischenlage für eine Mehrstöckige Torte würde ich dies jedoch nicht verwenden.","produkt ist wie beschrieben. gute qualität, gute optik. als zwischenlage für eine mehrstöckige torte würde ich dies jedoch nicht verwenden.","Title: Alles super Review: Produkt ist wie beschrieben. Gute Qualität, gute Optik. Als Zwischenlage für eine Mehrstöckige Torte würde ich dies jedoch nicht verwenden.","Title: Alles super Review: Produkt ist wie beschrieben. Gute Qualität, gute Optik. Als Zwischenlage für 

## URL preservation check

In [10]:
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

cfg = Preprocessor_Configs(
    preprocessing_level='light',
    input_format='review_only',
    truncation=True,
    max_length=128,
    padding=False,
)

preproc = Preprocessor(cfg, tokenizer, return_special_tokens_mask=False)

url_examples = df[df['sentence'].astype(str).str.contains(r'https?://|www\.', regex=True, na=False)].head(10)

for i, text in enumerate(url_examples['sentence'].tolist(), start=1):
    processed = preproc.preprocess_one_text(text)
    print('=' * 100)
    print(f'URL EXAMPLE {i}')
    print('=' * 100)
    print('RAW:')
    print(text)
    print()
    print('PROCESSED:')
    print(processed)
    print()

URL EXAMPLE 1
RAW:
Stabil passt

Verschlüsse alle gut.. Material etwas hart aber macht dadurch ein stabilen Eindruck.. Platz ist super.. Innen Tasche... Alles stabil vernäht.. Nur störend www. Werbung an einem Träger.

PROCESSED:
Verschlüsse alle gut.. Material etwas hart aber macht dadurch ein stabilen Eindruck.. Platz ist super.. Innen Tasche... Alles stabil vernäht.. Nur störend www. Werbung an einem Träger.

URL EXAMPLE 2
RAW:
Not for me

these are uncomfortable, they are thick, hard and skinny, ewww.

PROCESSED:
these are uncomfortable, they are thick, hard and skinny, ewww.

URL EXAMPLE 3
RAW:
geht es nicht die Kabel noch kürzer zu machen ?

Ich habe das Ding heute eingebaut , so lange das Antennenkabel frei im Auto lag hatte ich 56 Stationen , super . Dann habe ich das Antennenkabel verlegt , das ist übrigens das Kabel was lang genug ist. So alles schön versteckt das DAB Ding an , nur noch eine Station , kann mir das mal einer erklären bitte . Das ging ja schon los mit dem Platz

## Length comparison

In [11]:
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

stats = []

for _, row in df_sample.iterrows():
    raw_text = row['sentence']

    for input_format in ['review_only', 'title_plus_review']:
        for level in ['light', 'medium', 'heavy']:
            cfg = Preprocessor_Configs(
                preprocessing_level=level,
                input_format=input_format,
                truncation=True,
                max_length=128,
                padding=False,
            )
            preproc = Preprocessor(cfg, tokenizer, return_special_tokens_mask=False)
            processed = preproc.preprocess_one_text(raw_text)

            stats.append({
                'input_format': input_format,
                'level': level,
                'char_len': len(processed),
                'word_len': len(processed.split())
            })

stats_df = pd.DataFrame(stats)
stats_df.groupby(['input_format', 'level'])[['char_len', 'word_len']].mean().round(2)

char_len  word_len
input_format      level                     
review_only       heavy     173.30     28.07
                  light     173.70     28.07
                  medium    173.67     28.07
title_plus_review heavy     211.00     33.60
                  light     211.57     33.63
                  medium    211.53     33.63

## Tokenization sanity check

In [13]:
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')

example = manual_examples[0]

for level in ['light', 'medium', 'heavy']:
    cfg = Preprocessor_Configs(
        preprocessing_level=level,
        input_format='title_plus_review',
        truncation=True,
        max_length=128,
        padding=False,
    )
    preproc = Preprocessor(cfg, tokenizer, return_special_tokens_mask=True)
    processed = preproc.preprocess_one_text(example)

    encoded = tokenizer(
        processed,
        truncation=True,
        max_length=128,
        padding=False,
        return_special_tokens_mask=True,
    )

    print('=' * 100)
    print(level.upper())
    print('=' * 100)
    print('Processed text:')
    print(processed)
    print()
    print('Tokens:')
    print(tokenizer.convert_ids_to_tokens(encoded['input_ids'][:40]))
    print()

LIGHT
Processed text:
Title: Great product Review: I absolutely loved it!!!!!!! 😊 Check URLTOKEN for details.

Tokens:
['<s>', '▁Title', ':', '▁Great', '▁product', '▁Review', ':', '▁I', '▁absolutely', '▁loved', '▁it', '!!!!!!!', '▁', '😊', '▁Check', '▁URL', 'TO', 'KEN', '▁for', '▁details', '.', '</s>']

MEDIUM
Processed text:
Title: Great product Review: I absolutely loved it!!! 😊 Check URLTOKEN for details.

Tokens:
['<s>', '▁Title', ':', '▁Great', '▁product', '▁Review', ':', '▁I', '▁absolutely', '▁loved', '▁it', '!!!', '▁', '😊', '▁Check', '▁URL', 'TO', 'KEN', '▁for', '▁details', '.', '</s>']

HEAVY
Processed text:
title: great product review: i absolutely loved it!! check urltoken for details.

Tokens:
['<s>', '▁title', ':', '▁great', '▁product', '▁review', ':', '▁i', '▁absolutely', '▁loved', '▁it', '!!', '▁check', '▁', 'url', 'tok', 'en', '▁for', '▁details', '.', '</s>']

